In [1]:
from pathlib import Path
import numpy as np
import sys

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from spike_classifier.annotate_spikes import annotate_spikes
from spike_classifier.train_classifier import train_spike_classifier
from spike_classifier.prepare_data import prepare_spike_data
from utils.label_utils import reset_spike_labels


In [2]:
FS = 3

ROI_DATA_PATH = PROJECT_ROOT / "data" / "invivo_roi_features_z_humanITL.npy"
assert ROI_DATA_PATH.exists(), f"Data file not found: {ROI_DATA_PATH}"
SPIKE_OUTPUT_PATH = PROJECT_ROOT / "data" / "invivo_roi_features_z_humanITL_diff-spikes.npy"
MODEL_OUT_DIR = PROJECT_ROOT / "models"
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = PROJECT_ROOT / "config" / "classifier_config.yaml"

print(f"Path to data: {ROI_DATA_PATH}")
print(f"Saving models to: {MODEL_OUT_DIR}")
print(f"Config: {CONFIG_PATH}")

Path to data: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\invivo_roi_features_z_humanITL.npy
Saving models to: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models
Config: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\classifier_config.yaml


In [3]:

reset = False # Gated purposely to avoid accidental resets
if reset:
    roi_dict = np.load(ROI_DATA_PATH, allow_pickle=True).item()
    roi_dict, n_reset = reset_spike_labels(roi_dict)
    np.save(ROI_DATA_PATH, roi_dict, allow_pickle=True)
    print(f"Reset {n_reset} labels to unlabeled")

In [3]:

roi_dict = prepare_spike_data(
    input_path=str(SPIKE_OUTPUT_PATH),
    output_path=None,  
    max_rois=None,
    fs=FS             # Frame rate in Hz — adjust to match your acquisition rate
)


Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\invivo_roi_features_z_humanITL_diff-spikes.npy

  SPIKE Summary
  ROIs: 1942  (196 with spikes)
  Total spikes: 6538
  Good: 250 | Bad: 701 | Unlabeled: 5587
  Manual: 951 | Auto: 0



In [9]:
n_samples = 5000
unlabeled_only = False
labeled_only = False
checkpoint_interval = 1000

annotate_spikes(
    data_path=SPIKE_OUTPUT_PATH,
    max_rois=n_samples,
    unlabeled_only=unlabeled_only,
    labeled_only=labeled_only,
    checkpoint_interval=checkpoint_interval,
    verbose=True
)


Session ended by user. Saving progress...
Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\invivo_roi_features_z_humanITL_diff-spikes.npy
Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\invivo_roi_features_z_humanITL_diff-spikes.npy

  SPIKE Annotation Summary
  Queued:    6538
  Seen:      866
  Labeled:   866
  Updated:   837
  Confirmed: 29
  Skipped:   0


  SPIKE Summary
  ROIs: 1942  (196 with spikes)
  Total spikes: 6538
  Good: 250 | Bad: 701 | Unlabeled: 5587
  Manual: 951 | Auto: 0



{'level': 'spike',
 'queued': 6538,
 'total': 866,
 'labeled': 866,
 'updated': 837,
 'confirmed': 29,
 'skipped': 0,
 'queued_rois': 196}

In [4]:

name = "invivo_spike_classifier_z_diff_1" # TODO Change as desired for your organizational needs

results = train_spike_classifier(
    config_path=CONFIG_PATH,
    data_path=SPIKE_OUTPUT_PATH,
    name=name,
    output_dir=MODEL_OUT_DIR,
    verbose=True,
    manual_only=True
)


Dataset Summary
--------------------------------------------------
Total labeled datapoints: 951
  Train: 760 | Test: 191

Label distribution:
              Bad (0)  Good (1)
  Train           555       205
  Test            146        45
  Total           701       250

Training on: Manual labels only

--------------------------------------------------
TUNED MODEL SUMMARY
--------------------------------------------------
Model:     LogisticRegression
Transform: sqrt
Features:  ['raw_fluorescence_value', 'value', 'spike_prom', 'distance', 'mini_prom']

Hyperparameters:
  C: 10
  class_weight: balanced
  max_iter: 200
  penalty: l1
  solver: liblinear

Metrics:
  CV Accuracy:   0.9000
  Test Accuracy: 0.8848
  ROC AUC:       0.9680
  F1:            0.8902
  Precision:     0.9090
  Recall:        0.8848

Confusion Matrix:
              Pred 0  Pred 1
  Actual 0    127     19     
  Actual 1    3       42     
--------------------------------------------------
Saved model to C:\Users\mzi